# Multi-Session vmPFC θ SFC & Hipp γ↔vmPFC θ ir-PAC
Pipeline that quantifies hippocampal spike-to-vmPFC theta phase locking and hippocampal gamma amplitude coupling to vmPFC theta phase across multiple sessions.

## 1. Setup & Imports

In [51]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

sns.set(style="whitegrid", context="talk")
warnings.filterwarnings("ignore", category=RuntimeWarning)

from nwb_analysis import get_subject_files
from nwb_analysis.cfc import (
    prepare_session_structures,
    compute_vmPFC_theta_phase,
    compute_hipp_gamma_envelope,
    compute_irpac,
    run_session_stats,
    plot_session_summary,
    save_session_outputs,
)


## 2. Config & Session Selection

In [ ]:
# Session + storage config
DATA_DIR = Path('E:/SBCAT/data/000673')
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)
USE_ONLY_CORRECT = True

# Session selection: set to None to process all sessions, or specify filters
SUBJECT_FILTER = [21,22,23,24,25,26,27,28,29,30]  # e.g., [1, 2, 3] to process only sub-1, sub-2, sub-3
SESSION_PATTERN = None  # e.g., 'ses-1' to process only session 1 files
MAX_SESSIONS = None  # Limit number of sessions to process (useful for testing)

CONDITIONS = ['L1', 'L3']
EPOCH_ANALYZE = (0, 2.5)       # s window used for analyses relative to maintenance onset
EPOCH_LABEL = 'maintenance'
THETA_BAND = (3.0, 7.0)
PHASE_BAND_LABEL = 'theta'
GAMMA_BAND = (30.0, 140.0)
GAMMA_BAND_LABEL = 'gamma'
CAR_MODE = 'hemisphere'
MIN_TRIALS_PER_COND = 20
PAC_SURR_N = 200
LAG_GRID = np.arange(-0.150, 0.155, 0.005)  # seconds
EXCLUDE_LAG_S = (0.010, 0.020)              # ignore central lags when reporting lag-swept PAC
PEAK_TO_PEAK_THRESH = 3000.0  # µV; set to None to skip automatic rejection
PAC_ALPHA = 0.05

ANALYSIS_PARAMS = {
    'conditions': CONDITIONS,
    'epoch_analyze': EPOCH_ANALYZE,
    'epoch_label': EPOCH_LABEL,
    'theta_band': THETA_BAND,
    'phase_band_label': PHASE_BAND_LABEL,
    'gamma_band': GAMMA_BAND,
    'gamma_band_label': GAMMA_BAND_LABEL,
    'car_mode': CAR_MODE,
    'min_trials': MIN_TRIALS_PER_COND,
    'pac_surrogates': PAC_SURR_N,
    'lag_grid_s': LAG_GRID.tolist(),
    'exclude_lag_s': EXCLUDE_LAG_S,
    'max_peak_to_peak': PEAK_TO_PEAK_THRESH,
    'pac_alpha': PAC_ALPHA,
}

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUT_DIR}")


Data directory: E:/SBCAT/data/000673
Output directory: outputs


### Discover and filter session files

In [53]:
from pathlib import Path
import re


def get_sessions_to_process(data_dir: Path, subject_filter=None, session_pattern=None, max_sessions=None):
    '''Get list of NWB session files to process with optional filtering.'''
    all_files = get_subject_files(data_dir)
    
    if not all_files:
        raise FileNotFoundError(f"No NWB files found under {data_dir}")
    
    # Filter by subject ID if specified
    if subject_filter is not None:
        if isinstance(subject_filter, (int, str)):
            subject_filter = [subject_filter]
        # Convert to strings and create exact match patterns
        subject_patterns = [f"sub-{sub_id}/" for sub_id in subject_filter]
        filtered = []
        for f in all_files:
            f_str = str(f)
            for pattern in subject_patterns:
                if pattern in f_str:
                    filtered.append(f)
                    break
        all_files = filtered
    
    # Filter by session pattern if specified
    if session_pattern is not None:
        all_files = [f for f in all_files if session_pattern in str(f)]
    
    # Limit number of sessions if specified
    if max_sessions is not None:
        all_files = all_files[:max_sessions]
    
    return sorted(all_files)


# Get all sessions to process
all_sessions = get_sessions_to_process(
    DATA_DIR, 
    subject_filter=SUBJECT_FILTER,
    session_pattern=SESSION_PATTERN,
    max_sessions=MAX_SESSIONS
)

print(f"Found {len(all_sessions)} session file(s) to process:")
for i, session_path in enumerate(all_sessions, 1):
    print(f"  {i}. {session_path.relative_to(DATA_DIR)}")


Found 13 session file(s) to process:
  1. sub-21/sub-21_ses-1_ecephys+image.nwb
  2. sub-22/sub-22_ses-1_ecephys+image.nwb
  3. sub-22/sub-22_ses-2_ecephys+image.nwb
  4. sub-22/sub-22_ses-3_ecephys+image.nwb
  5. sub-23/sub-23_ses-1_ecephys+image.nwb
  6. sub-24/sub-24_ses-1_ecephys+image.nwb
  7. sub-25/sub-25_ses-1_ecephys+image.nwb
  8. sub-26/sub-26_ses-1_ecephys+image.nwb
  9. sub-26/sub-26_ses-2_ecephys+image.nwb
  10. sub-27/sub-27_ses-1_ecephys+image.nwb
  11. sub-28/sub-28_ses-1_ecephys+image.nwb
  12. sub-29/sub-29_ses-1_ecephys+image.nwb
  13. sub-30/sub-30_ses-1_ecephys+image.nwb


## 3. Multi-Session Processing Loop

In [54]:
# Track processing results
processing_results = []
failed_sessions = []

# Process each session
for session_idx, session_path in enumerate(tqdm(all_sessions, desc="Processing sessions"), 1):
    try:
        # Extract session ID from path (relative to DATA_DIR)
        session_id = str(session_path.relative_to(DATA_DIR))
        
        # Create session-specific output directory
        session_out_dir = OUT_DIR / session_id / 'sfc_irpac_single_session'
        session_out_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"\n{'='*80}")
        print(f"Processing session {session_idx}/{len(all_sessions)}: {session_id}")
        print(f"{'='*80}")
        
        # Load session data
        session_data = prepare_session_structures(
            session_id=session_id,
            session_path=session_path,
            conditions=CONDITIONS,
            use_only_correct=USE_ONLY_CORRECT,
            epoch_label=EPOCH_LABEL,
        )
        
        trial_table = session_data['trial_table']
        spike_times_by_unit = session_data['spike_times_by_unit']
        unit_meta = session_data['unit_meta']
        lfp_by_channel = session_data['lfp_by_channel']
        lfp_fs = session_data['lfp_fs']
        lfp_start_time = session_data['lfp_start_time']
        channel_meta = session_data['channel_meta']
        session_meta = session_data['session_meta']
        
        print(f"Trials: {len(trial_table)} | Units: {len(spike_times_by_unit)} | Channels: {len(lfp_by_channel)} @ {lfp_fs:.1f} Hz")
        
        # Preprocessing
        theta_phase = compute_vmPFC_theta_phase(
            lfp_by_channel=lfp_by_channel,
            channel_meta=channel_meta,
            trial_table=trial_table,
            sampling_rate=lfp_fs,
            lfp_start_time=lfp_start_time,
            epoch_analyze=EPOCH_ANALYZE,
            theta_band=THETA_BAND,
            car_mode=CAR_MODE,
            trial_pad_sec=0.5,
            filter_order=4,
        )
        
        hipp_gamma_amp = compute_hipp_gamma_envelope(
            lfp_by_channel=lfp_by_channel,
            channel_meta=channel_meta,
            trial_table=trial_table,
            sampling_rate=lfp_fs,
            lfp_start_time=lfp_start_time,
            epoch_analyze=EPOCH_ANALYZE,
            gamma_band=GAMMA_BAND,
            trial_pad_sec=0.5,
            filter_order=4,
        )
        
        print(f"vmPFC θ channels: {len(theta_phase)}")
        print(f"Hipp γ channels: {len(hipp_gamma_amp)}")
        
        # LFP Pair Formation
        hipp_channels = channel_meta[channel_meta['area'].str.contains('hipp', case=False, na=False)].copy()
        vm_channels = channel_meta[channel_meta['area'].str.contains('vmpfc', case=False, na=False)].copy()
        hipp_channels['hemisphere'] = hipp_channels.get('hemisphere', 'unknown').fillna('unknown')
        vm_channels['hemisphere'] = vm_channels.get('hemisphere', 'unknown').fillna('unknown')
        
        pairs = []
        for hipp in hipp_channels.itertuples():
            for vm in vm_channels.itertuples():
                if hipp.hemisphere != 'unknown' and vm.hemisphere != 'unknown' and hipp.hemisphere != vm.hemisphere:
                    continue
                pair_id = f"hipp{hipp.chan_id}_vm{vm.chan_id}"
                pairs.append({
                    'pair_id': pair_id,
                    'unit_id': hipp.chan_id,
                    'hipp_chan_id': hipp.chan_id,
                    'unit_channel': hipp.chan_id,
                    'hipp_hemisphere': hipp.hemisphere,
                    'chan_id': vm.chan_id,
                    'vm_hemisphere': vm.hemisphere,
                })
        
        pair_table = pd.DataFrame(pairs)
        print(f"Candidate hipp×vmPFC channel pairs: {len(pair_table)}")
        
        # Compute ir-PAC
        pac_results = compute_irpac(
            pair_table=pair_table[:],
            trial_table=trial_table,
            theta_phase=theta_phase,
            gamma_envelope=hipp_gamma_amp,
            lfp_fs=lfp_fs,
            epoch_analyze=EPOCH_ANALYZE,
            conditions=CONDITIONS,
            min_trials=MIN_TRIALS_PER_COND,
            n_surrogates=PAC_SURR_N,
            lag_grid_s=None,
            exclude_lag_s=EXCLUDE_LAG_S,
            significance_alpha=PAC_ALPHA,
            phase_band=THETA_BAND,
            amp_band=GAMMA_BAND,
            phase_band_label=PHASE_BAND_LABEL,
            amp_band_label=GAMMA_BAND_LABEL,
            epoch_label=EPOCH_LABEL,
        )
        
        # Session-level stats
        stats_summary = run_session_stats(
            sfc_results=None,
            pac_results=pac_results,
            conditions=CONDITIONS,
        )
        
        # QC & Plots
        summary_figs = plot_session_summary(
            sfc_results=None,
            pac_results=pac_results,
            stats_summary=stats_summary,
            conditions=CONDITIONS,
            output_dir=session_out_dir,
            dpi=150,
        )
        
        print(f"✓ Session {session_idx} complete. Outputs saved to: {session_out_dir}")
        print(f"  Summary figures: {[str(f.name) for f in summary_figs]}")
        
        # Store results
        processing_results.append({
            'session_id': session_id,
            'session_path': str(session_path),
            'output_dir': str(session_out_dir),
            'n_trials': len(trial_table),
            'n_units': len(spike_times_by_unit),
            'n_channels': len(lfp_by_channel),
            'n_pairs': len(pair_table),
            'stats_summary': stats_summary,
            'success': True,
        })
        
    except Exception as e:
        error_msg = f"Error processing {session_path.name}: {str(e)}"
        print(f"✗ {error_msg}")
        failed_sessions.append({
            'session_path': str(session_path),
            'error': str(e),
        })
        import traceback
        traceback.print_exc()
        continue

print(f"\n{'='*80}")
print(f"Processing complete!")
print(f"  Successful: {len(processing_results)}/{len(all_sessions)}")
print(f"  Failed: {len(failed_sessions)}/{len(all_sessions)}")
if failed_sessions:
    print(f"\nFailed sessions:")
    for fail in failed_sessions:
        print(f"  - {fail['session_path']}: {fail['error']}")


Processing sessions:   0%|          | 0/13 [00:00<?, ?it/s]


Processing session 1/13: sub-21/sub-21_ses-1_ecephys+image.nwb


/Users/jundazhu/.pyenv/versions/3.10.17/lib/python3.10/site-packages/pynwb/ecephys.py:158: UserWarning: ElectricalSeries 'LFPs': The second dimension of data does not match the length of electrodes. Your data may be transposed.
  warnings.warn("%s '%s': The second dimension of data does not match the length of electrodes. "


Trials: 136 | Units: 40 | Channels: 56 @ 400.0 Hz


Processing sessions:   8%|▊         | 1/13 [00:02<00:32,  2.71s/it]

...

✓ Session 12 complete. Outputs saved to: outputs/sub-29/sub-29_ses-1_ecephys+image.nwb/sfc_irpac_single_session
  Summary figures: ['pac_session_summary.png']

Processing session 13/13: sub-30/sub-30_ses-1_ecephys+image.nwb


/Users/jundazhu/.pyenv/versions/3.10.17/lib/python3.10/site-packages/pynwb/ecephys.py:158: UserWarning: ElectricalSeries 'LFPs': The second dimension of data does not match the length of electrodes. Your data may be transposed.
  warnings.warn("%s '%s': The second dimension of data does not match the length of electrodes. "


Trials: 139 | Units: 16 | Channels: 33 @ 400.0 Hz


Processing sessions: 100%|██████████| 13/13 [01:16<00:00,  5.86s/it]

vmPFC θ channels: 0
Hipp γ channels: 19
Candidate hipp×vmPFC channel pairs: 0
✓ Session 13 complete. Outputs saved to: outputs/sub-30/sub-30_ses-1_ecephys+image.nwb/sfc_irpac_single_session
  Summary figures: []

Processing complete!
  Successful: 13/13
  Failed: 0/13


## 4. Processing Summary

In [55]:
# Display processing summary
if processing_results:
    summary_df = pd.DataFrame([
        {
            'session_id': r['session_id'],
            'n_trials': r['n_trials'],
            'n_units': r['n_units'],
            'n_channels': r['n_channels'],
            'n_pairs': r['n_pairs'],
            'output_dir': r['output_dir'],
        }
        for r in processing_results
    ])
    print("\nProcessing Summary:")
    print(summary_df.to_string(index=False))
else:
    print("No sessions were successfully processed.")


Processing Summary:
                           session_id  n_trials  n_units  n_channels  n_pairs                                                             output_dir
sub-21/sub-21_ses-1_ecephys+image.nwb       136       40          56        0 outputs/sub-21/sub-21_ses-1_ecephys+image.nwb/sfc_irpac_single_session
sub-22/sub-22_ses-1_ecephys+image.nwb       100       42          24        0 outputs/sub-22/sub-22_ses-1_ecephys+image.nwb/sfc_irpac_single_session
sub-22/sub-22_ses-2_ecephys+image.nwb       128       13          38        0 outputs/sub-22/sub-22_ses-2_ecephys+image.nwb/sfc_irpac_single_session
sub-22/sub-22_ses-3_ecephys+image.nwb       127       13          38        0 outputs/sub-22/sub-22_ses-3_ecephys+image.nwb/sfc_irpac_single_session
sub-23/sub-23_ses-1_ecephys+image.nwb       131       18          33        0 outputs/sub-23/sub-23_ses-1_ecephys+image.nwb/sfc_irpac_single_session
sub-24/sub-24_ses-1_ecephys+image.nwb       117        7          14        0 outputs